# Modul 15: TensorFlow-Grundlagen und dichte Keras-Modelle | Übungen

## Überblick

Sie untersuchen TensorFlow-Tensoren, Broadcasting, automatische Gradienten und tf.data-Pipelines. Danach erstellen, trainieren, regularisieren, bewerten und speichern Sie ein kleines dichtes Keras-Modell für tabellarische Daten.

**Zugehörige Vorlesungen**

- **TensorFlow Grundlagen**
- **Dichte Keras-Modelle**

## Lernziele

Nach der Bearbeitung können Sie:

- TensorFlow-Tensoren, Formen, Datentypen, NumPy-Konvertierung und Broadcasting sicher verwenden.
- Gradienten mit GradientTape berechnen und kleine tf.data-Datasets reproduzierbar verarbeiten.
- Sequential-Modelle mit passenden Ein- und Ausgaben kompilieren, trainieren, bewerten und speichern.

## Geprüfte Fähigkeiten

- Tensoren, Datentypen, Broadcasting und GradientTape
- tf.data mit shuffle, batch, map und prefetch
- Keras Sequential, compile, fit, History, Regularisierung, Baseline und Modellpersistenz

## Hinweise zur Bearbeitung

Dieses Notebook dient als praktische Übung und Lernstandskontrolle. Führen Sie zuerst die Einrichtungszelle aus und bearbeiten Sie danach die Aufgaben in der angegebenen Reihenfolge. Die vorgesehenen Arbeitsbereiche sind deutlich markiert.

- **Erwarteter Schwierigkeitsgrad:** fortgeschritten
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle prüft TensorFlow und lädt den kleinen Brustkrebs-Datensatz aus scikit-learn. Die Daten werden reproduzierbar in Training, Validierung und Test geteilt und ohne Test-Leakage skaliert. Das Modell bleibt klein und läuft auf der kostenlosen Colab-CPU.

In [ ]:
import os
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.dummy import DummyClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

print("TensorFlow-Version:", tf.__version__)
print("Verfügbare GPUs:", tf.config.list_physical_devices("GPU"))

daten = load_breast_cancer()
X_gesamt = daten.data.astype("float32")
y_gesamt = daten.target.astype("float32")

X_train_roh, X_test_roh, y_train, y_test = train_test_split(
    X_gesamt,
    y_gesamt,
    test_size=0.20,
    stratify=y_gesamt,
    random_state=RANDOM_SEED,
)
X_train_roh, X_val_roh, y_train, y_val = train_test_split(
    X_train_roh,
    y_train,
    test_size=0.20,
    stratify=y_train,
    random_state=RANDOM_SEED,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_roh).astype("float32")
X_val = scaler.transform(X_val_roh).astype("float32")
X_test = scaler.transform(X_test_roh).astype("float32")

print("Train, Validierung, Test:", X_train.shape, X_val.shape, X_test.shape)

### Aufgabe 1: Tensoren, Formen, Datentypen und NumPy-Konvertierung

Erzeugen Sie aus der vorgegebenen Python-Liste einen Tensor mit `dtype=tf.float32`. Geben Sie Form, Rang und Datentyp aus. Wandeln Sie ihn in ein NumPy-Array zurück und prüfen Sie Werte und Form.

Erzeugen Sie außerdem einen Integer-Tensor und zeigen Sie, dass für eine Division zunächst eine explizite Typumwandlung sinnvoll ist.

In [ ]:
python_matrix = [[1.0, 2.5, -1.0], [0.0, 4.0, 3.5]]

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum sollte man Form und Datentyp bereits vor dem Modelltraining prüfen?

### Aufgabe 2: Broadcasting mit TensorFlow nachvollziehen

Erzeugen Sie einen Tensor `bilder_batch` der Form `(2, 3, 4, 3)`, der zwei kleine RGB-Bilder repräsentiert. Ziehen Sie den Kanalvektor `[10, 20, 30]` per Broadcasting von jedem Pixel ab.

Prüfen Sie die resultierende Form und vergleichen Sie einen ausgewählten Pixel vor und nach der Operation. Erzeugen Sie außerdem ein absichtlich inkompatibles Gegenbeispiel und fangen Sie den erwarteten Fehler mit `try/except` ab.

In [ ]:
bilder_batch = tf.reshape(tf.range(2 * 3 * 4 * 3, dtype=tf.float32), (2, 3, 4, 3))
kanal_mittel = tf.constant([10.0, 20.0, 30.0], dtype=tf.float32)

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Wie werden Broadcasting-Formen geprüft?

### Aufgabe 3: GradientTape mit manuellem Gradienten vergleichen

Für die Funktion

`L(w) = mean((w * x - y)^2)`

soll der Gradient nach dem skalaren Gewicht `w` berechnet werden. Verwenden Sie `tf.GradientTape` und leiten Sie den Gradienten zusätzlich mit NumPy manuell her. Führen Sie danach fünf Gradientenschritte mit Lernrate 0.1 aus und speichern Sie Gewicht und Verlust nach jedem Schritt.

In [ ]:
x_grad = tf.constant([1.0, 2.0, 3.0, 4.0], dtype=tf.float32)
y_grad = tf.constant([2.0, 4.0, 6.0, 8.0], dtype=tf.float32)
w = tf.Variable(0.5, dtype=tf.float32)

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum muss ein trainierbares Gewicht als tf.Variable statt nur als tf.constant angelegt werden?

### Aufgabe 4: tf.data mit Shuffling, Mapping und Batching aufbauen

Erstellen Sie aus `X_train` und `y_train` ein `tf.data.Dataset`. Mischen Sie mit festem Seed, verwenden Sie Batchgröße 32 und wenden Sie per `map` eine Funktion an, die jedem Merkmalsvektor eine zusätzliche letzte Achse gibt und das Label in `float32` umwandelt.

Nutzen Sie `prefetch(tf.data.AUTOTUNE)`. Inspizieren Sie Formen und Datentypen des ersten Batches. Erstellen Sie außerdem geordnete Validierungs- und Test-Datasets ohne Shuffling, jedoch ohne zusätzliche Achse, damit sie später zum dichten Modell passen.

In [ ]:
def erweitere_beispiel(merkmale, label):
    pass

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum sollte das Validierungs- und Test-Dataset normalerweise nicht gemischt werden?

### Aufgabe 5: Ein passendes Sequential-Modell erstellen und trainieren

Erstellen Sie ein kleines binäres Sequential-Modell mit expliziter Eingabeform, zwei Dense-Schichten und einer Sigmoid-Ausgabe. Verwenden Sie höchstens 16 und 8 verborgene Einheiten.

Kompilieren Sie mit Adam, binärer Kreuzentropie, Accuracy sowie AUC. Trainieren Sie höchstens 40 Epochen auf `train_dataset`, verwenden Sie `val_dataset` und EarlyStopping mit Wiederherstellung der besten Gewichte. Zeichnen Sie Trainings- und Validierungsverlust und vergleichen Sie Test-Accuracy sowie F1 mit einer Mehrheitsbaseline.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum passen eine Sigmoid-Ausgabe und binary_crossentropy zusammen?

### Aufgabe 6: Integrationsaufgabe: Regularisierung, Vorhersagen und Speichern

Erstellen Sie ein zweites Modell mit derselben Grundstruktur, ergänzen Sie aber L2-Regularisierung in den Dense-Schichten und Dropout nach der ersten verborgenen Schicht. Trainieren Sie mit denselben Daten und EarlyStopping.

Vergleichen Sie Parameterzahl, besten Validierungsverlust und Test-F1 beider Modelle. Speichern Sie das bessere Modell in einem temporären Verzeichnis im `.keras`-Format, laden Sie es neu und prüfen Sie, ob die ersten zehn Wahrscheinlichkeiten vor und nach dem Laden übereinstimmen.

In [ ]:
from tensorflow.keras import regularizers

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Welche zusätzlichen Objekte müssen zusammen mit einem produktiven Modell dokumentiert oder gespeichert werden?

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?